In [ ]:
%cd /content
!git clone https://github.com/madelyn-redick/LearningASL.git
%cd /content/LearningASL

In [ ]:
# Create and switch to a new branch
!git checkout train-cnn

# Verify you're on the new branch
!git branch

In [ ]:
# Convert data_preprocessing.ipynb to python module
!jupyter nbconvert --to python data_preprocessing.ipynb --output data_preprocessing

In [ ]:
# Convert the CNN.ipynb to a .py file
!jupyter nbconvert --to python cnn/CNN.ipynb

In [ ]:
import sys
sys.path.append('/content/LearningASL')
import data_preprocessing

from cnn.CNN import ASL_CNN, train_model, evaluate_model

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

In [ ]:
# Load letter datasets
letter_train = data_preprocessing.letter_train
letter_val = data_preprocessing.letter_val
letter_test = data_preprocessing.letter_test

# Verify imported data
print(f"Type of letter_train: {type(letter_train)}")
print(f"Train: {len(letter_train)}, Val: {len(letter_val)}, Test: {len(letter_test)}")

In [ ]:
# Hyperparameter tuning
learning_rate = 1e-2 # small learning rate because we are using transfer learning and do not want to mess up pretrained weights
momentum = 0.9
lr_gamma = 0.9
epochs = 1
batch_size = 32
print_every = 10

In [ ]:
# Setup device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create DataLoader instances for each phase in training
dataloader = {
    'train': DataLoader(letter_train, batch_size=batch_size, shuffle=True, num_workers=2),
    'validation': DataLoader(letter_val, batch_size=batch_size, shuffle=False, num_workers=2)
}

# Create DataLoader for test set
test_dataloader = DataLoader(letter_test, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Model Instance
model = ASL_CNN(num_classes=28)

In [ ]:
# Setup training
# Optimize only the parameters that are not frozen (AKA requires_grad == True)
optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=learning_rate, momentum=momentum)
scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=lr_gamma)
loss_fn = nn.CrossEntropyLoss()

best_model_path = '/content/best_cnn_params.pth'

In [ ]:
# Train model
trained_model = train_model(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    dataloader=dataloader,
    device=device,
    best_model_path=best_model_path,
    num_epochs=epochs,
    print_every=print_every
)

In [ ]:
# Save best model to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_save_path = '/content/drive/MyDrive/LearningASL/models/'
!mkdir -p {drive_save_path}
shutil.copy(best_model_path, f'{drive_save_path}best_cnn_params.pth')
print(f"Model saved to Drive: {drive_save_path}")

In [ ]:
# Evaluate model with saved best params

model_path = '/content/drive/MyDrive/LearningASL/models/best_cnn_params.pth'
model.load_state_dict(torch.load(model_path, weights_only=True))

test_loss, test_accuracy = evaluate_model(
    model=trained_model,
    loss_fn=loss_fn,
    test_dataloader=test_dataloader,
    device=device,
    print_every=print_every
)

print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy:.4f}")